In [5]:
import torch
import torch.nn as nn

from datasets import load_dataset
from collections import Counter

### 1.2 Datensatz laden

Für das Training und die Bewertung des Modells wird der ART-Datensatz geladen.

Der Datensatz ist in drei Bereiche aufgeteilt:

- **Training:** Daten, mit denen das Modell lernt.
- **Validation:** Daten, mit denen die Modellleistung während der Entwicklung überprüft wird.
- **Test:** Daten für die abschliessende Bewertung des fertigen Modells.

Der Testdatensatz wird erst nach Abschluss der Modellentwicklung verwendet.

In [6]:
train = load_dataset("amarfurt/art-nlp-hs26", split="train")
valid = load_dataset("amarfurt/art-nlp-hs26", split="validation")

print("Trainingsbeispiele:", len(train))
print("Validierungsbeispiele:", len(valid))

Trainingsbeispiele: 16801
Validierungsbeispiele: 1000


### 1.3 Tokenisierung

Die Tokenisierung erfolgt über eine wiederverwendbare Funktion aus `src/preprocessing.py`. Dabei werden Texte in Kleinbuchstaben umgewandelt und Wörter sowie Satzzeichen als separate Tokens behandelt.

In [7]:
import sys
from pathlib import Path

# Projektverzeichnis zum Python-Suchpfad hinzufügen
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing import tokenize

# Funktion anhand eines Trainingsbeispiels testen
example = train[0]

print("Originaltext:", example["observation_1"])
print("Tokens:", tokenize(example["observation_1"]))

Originaltext: A baby panda was lost in the woods.
Tokens: ['a', 'baby', 'panda', 'was', 'lost', 'in', 'the', 'woods', '.']


### 1.4 Vokabular erstellen

Das Vokabular wird ausschliesslich aus den Trainingsdaten erstellt. Jedes enthaltene Token erhält eine eindeutige numerische ID.

Die speziellen Tokens `<PAD>` und `<UNK>` werden für Padding beziehungsweise unbekannte Wörter verwendet.

Über den Parameter `min_freq` kann festgelegt werden, wie häufig ein Token mindestens vorkommen muss, um ins Vokabular aufgenommen zu werden.

In [8]:
import importlib
import src.preprocessing

importlib.reload(src.preprocessing)

<module 'src.preprocessing' from 'C:\\Users\\Marco\\Marco\\Github\\NLP-Project-1\\src\\preprocessing.py'>

In [9]:
from src.preprocessing import build_vocab

vocab = build_vocab(train, min_freq=1)

print("Vokabulargrösse:", len(vocab))
print("ID von 'panda':", vocab["panda"])
print("ID von '<PAD>':", vocab["<PAD>"])
print("ID von '<UNK>':", vocab["<UNK>"])

Vokabulargrösse: 15818
ID von 'panda': 4
ID von '<PAD>': 0
ID von '<UNK>': 1


### 1.5 Kodierung der Texte

Die tokenisierten Texte werden mithilfe des Vokabulars in numerische Token-IDs umgewandelt. Für Tokens, die nicht im Vokabular enthalten sind, wird die spezielle ID `<UNK>` verwendet.

Dadurch können auch Texte verarbeitet werden, die bisher unbekannte Wörter enthalten.

In [10]:
import importlib
import src.preprocessing

importlib.reload(src.preprocessing)

from src.preprocessing import encode

example = train[0]

token_ids = encode(example["observation_1"], vocab)

print("Originaltext:", example["observation_1"])
print("Token-IDs:", token_ids)
print("Anzahl Tokens:", len(token_ids))

Originaltext: A baby panda was lost in the woods.
Token-IDs: [2, 3, 4, 5, 6, 7, 8, 9, 10]
Anzahl Tokens: 9


### 1.6 Trainingsbeispiele vorbereiten

Jedes Trainingsbeispiel enthält zwei Beobachtungen, zwei Hypothesen und ein Label, das die korrekte Hypothese angibt.

Die vier Texte werden mithilfe der Funktion `encode()` in Token-IDs umgewandelt.

Da die Labels im Datensatz die Werte 1 und 2 besitzen, werden sie für die spätere Verwendung mit PyTorch in 0 und 1 umgewandelt.

In [11]:
import importlib
import src.preprocessing

# Aktualisierte Python-Datei neu laden
importlib.reload(src.preprocessing)

from src.preprocessing import encode_example

encoded_example = encode_example(train[0], vocab)

for key, value in encoded_example.items():
    print(f"{key}: {value}")

observation_1: [2, 3, 4, 5, 6, 7, 8, 9, 10]
observation_2: [11, 5, 12, 13, 14, 15, 16, 17, 18, 19, 20, 8, 21, 10]
hypothesis_1: [11, 5, 22, 6, 10]
hypothesis_2: [23, 8, 24, 25, 8, 21, 26, 27, 28, 10]
label: 1


### 1.7 Batch-Verarbeitung und Padding

Für ein effizientes Training werden mehrere Trainingsbeispiele zu einem Batch zusammengefasst.

Da die Texte unterschiedlich viele Tokens enthalten, werden kürzere Sequenzen innerhalb eines Batches mit `<PAD>` aufgefüllt. Dadurch können sie gemeinsam als PyTorch-Tensor verarbeitet werden.

Die Padding-Tokens werden später beim Mean Pooling mithilfe einer Maske ignoriert.

In [12]:
from src.preprocessing import collate_batch

# Drei Trainingsbeispiele kodieren
examples = [
    encode_example(train[i], vocab)
    for i in range(3)
]

# Beispiele zu einem Batch zusammenfassen
batch = collate_batch(examples, pad_id=vocab["<PAD>"])

for key, value in batch.items():
    print(f"{key}: {value.shape}")

observation_1: torch.Size([3, 13])
observation_2: torch.Size([3, 14])
hypothesis_1: torch.Size([3, 11])
hypothesis_2: torch.Size([3, 11])
label: torch.Size([3])


### 1.8 DataLoader erstellen

Für das Training werden die kodierten Beispiele mithilfe eines PyTorch DataLoaders in Batches aufgeteilt.
Die Trainingsdaten werden vor jeder Epoche zufällig gemischt. Die Validierungsdaten bleiben in ihrer ursprünglichen Reihenfolge.
Die Funktion `collate_batch()` ergänzt unterschiedlich lange Sequenzen innerhalb eines Batches automatisch mit Padding.

In [13]:
from torch.utils.data import DataLoader

# Trainings- und Validierungsdaten kodieren
train_encoded = [encode_example(example, vocab) for example in train]
valid_encoded = [encode_example(example, vocab) for example in valid]

# Batchgrösse festlegen
batch_size = 32

# DataLoader erstellen
train_loader = DataLoader(
    train_encoded,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_batch
)

valid_loader = DataLoader(
    valid_encoded,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_batch
)

print("Trainings-Batches:", len(train_loader))
print("Validierungs-Batches:", len(valid_loader))

Trainings-Batches: 526
Validierungs-Batches: 32


## 2. Modellarchitektur

### 2.1 Word Embeddings

Das Modell verwendet eine trainierbare Embedding-Schicht, die jede Token-ID in einen Vektor mit 50 Dimensionen umwandelt.

Die Embeddings werden während des Trainings angepasst. Mithilfe von Masked Mean Pooling wird später für jeden Text ein einzelner Vektor berechnet, wobei Padding-Tokens ignoriert werden.

In [14]:
from src.models import WordEmbeddingModel

model = WordEmbeddingModel(
    vocab_size=len(vocab),
    embedding_dim=50
)

print(model)
print("Anzahl Parameter:", sum(p.numel() for p in model.parameters()))

WordEmbeddingModel(
  (embedding): Embedding(15818, 50, padding_idx=0)
  (classifier): Sequential(
    (0): Linear(in_features=150, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)
Anzahl Parameter: 800629


### 2.2 Masked Mean Pooling

Durch Mean Pooling werden die Embedding-Vektoren eines Textes zu einem einzigen Vektor zusammengefasst.

Damit künstlich hinzugefügte Padding-Tokens das Ergebnis nicht beeinflussen, wird eine Maske verwendet. Der Durchschnitt wird dadurch ausschliesslich über die tatsächlichen Tokens berechnet.

Das Ergebnis ist ein Vektor mit 50 Dimensionen pro Text.

In [15]:
import importlib
import src.models

importlib.reload(src.models)

from src.models import WordEmbeddingModel

model = WordEmbeddingModel(
    vocab_size=len(vocab),
    embedding_dim=50
)

# Ersten Trainings-Batch auswählen
batch = next(iter(train_loader))

# Mean Pooling für observation_1 testen
sentence_vectors = model.mean_pooling(batch["observation_1"])

print("Token-IDs:", batch["observation_1"].shape)
print("Satzvektoren:", sentence_vectors.shape)

Token-IDs: torch.Size([32, 15])
Satzvektoren: torch.Size([32, 50])


### 2.3 Kombination der Satzvektoren

Die beiden Beobachtungen und die beiden Hypothesen werden durch Masked Mean Pooling jeweils in einen Vektor mit 50 Dimensionen umgewandelt.

Anschliessend werden die Vektoren der beiden Beobachtungen mit jeweils einer Hypothese durch Concatenation verbunden.

Dadurch entstehen pro Trainingsbeispiel zwei Vektoren mit jeweils 150 Dimensionen, die später vom Klassifikator bewertet werden.

In [16]:
import importlib
import src.models

importlib.reload(src.models)

from src.models import WordEmbeddingModel

model = WordEmbeddingModel(
    vocab_size=len(vocab),
    embedding_dim=50
)

batch = next(iter(train_loader))

combination_1, combination_2 = model.combine_vectors(batch)

print("Kombination 1:", combination_1.shape)
print("Kombination 2:", combination_2.shape)

Kombination 1: torch.Size([32, 150])
Kombination 2: torch.Size([32, 150])


### 2.4 Klassifikator

Ein Feedforward Neural Network bewertet die beiden Kombinationen aus Beobachtungen und Hypothesen.

Der Klassifikator besteht aus zwei linearen Schichten mit einer ReLU-Aktivierungsfunktion dazwischen.

Beide Kombinationen durchlaufen denselben Klassifikator. Für jede Hypothese wird ein Score berechnet. Die beiden Scores dienen anschliessend als Grundlage für die Klassifikation.

In [17]:
import importlib
import src.models

importlib.reload(src.models)

from src.models import WordEmbeddingModel

model = WordEmbeddingModel(
    vocab_size=len(vocab),
    embedding_dim=50
)

batch = next(iter(train_loader))

combination_1, combination_2 = model.combine_vectors(batch)

score_1 = model.classifier(combination_1)
score_2 = model.classifier(combination_2)

print("Score 1:", score_1.shape)
print("Score 2:", score_2.shape)

print("\nBeispiel-Scores:")
print("Hypothese 1:", score_1[0].item())
print("Hypothese 2:", score_2[0].item())

Score 1: torch.Size([32, 1])
Score 2: torch.Size([32, 1])

Beispiel-Scores:
Hypothese 1: 0.1751750111579895
Hypothese 2: 0.21585428714752197


### 2.5 Forward Pass

Die `forward()`-Methode verbindet die einzelnen Verarbeitungsschritte des Modells.

Zunächst werden die Texte durch Word Embeddings und Masked Mean Pooling in Satzvektoren umgewandelt. Anschliessend werden die Beobachtungen mit den jeweiligen Hypothesen kombiniert und durch den gemeinsamen Klassifikator bewertet.

Das Modell gibt für jedes Trainingsbeispiel zwei Scores zurück. Der höhere Score entspricht der bevorzugten Hypothese.

In [18]:
import importlib
import src.models

importlib.reload(src.models)

from src.models import WordEmbeddingModel

model = WordEmbeddingModel(
    vocab_size=len(vocab),
    embedding_dim=50
)

batch = next(iter(train_loader))

# Vollständigen Forward Pass ausführen
logits = model(batch)

print("Logits:", logits.shape)
print("\nErste drei Beispiele:")
print(logits[:3])

# Vorhersagen bestimmen
predictions = torch.argmax(logits, dim=1)

print("\nVorhersagen:", predictions[:3])
print("Tatsächliche Labels:", batch["label"][:3])

Logits: torch.Size([32, 2])

Erste drei Beispiele:
tensor([[ 0.0376,  0.0590],
        [-0.0186, -0.0189],
        [-0.0811, -0.0987]], grad_fn=<SliceBackward0>)

Vorhersagen: tensor([1, 0, 0])
Tatsächliche Labels: tensor([0, 1, 1])


### 2.6 Loss-Funktion

Für die Klassifikation wird `CrossEntropyLoss` verwendet.

Die Funktion vergleicht die beiden vom Modell berechneten Logits mit dem tatsächlichen Label (0 oder 1).

Ein niedriger Loss bedeutet, dass das Modell dem korrekten Label eine hohe Wahrscheinlichkeit zuordnet. Während des Trainings wird versucht, diesen Loss zu minimieren.

In [19]:
# Loss-Funktion definieren
criterion = nn.CrossEntropyLoss()

# Loss für den aktuellen Batch berechnen
loss = criterion(logits, batch["label"])

print("Loss:", loss.item())

Loss: 0.6903042197227478


### 2.7 Optimizer

Für das Training wird der AdamW-Optimizer mit einer Learning Rate von 0.001 verwendet.
Der Optimizer aktualisiert die trainierbaren Parameter des Modells anhand der berechneten Gradienten.
Dadurch können sowohl die Word Embeddings als auch die Gewichte des Klassifikators während des Trainings angepasst werden.

In [20]:
# Optimizer definieren
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

print(optimizer)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.01
)


### 2.8 Erster Trainingsschritt

In einem Trainingsschritt werden zunächst die Vorhersagen und der Loss berechnet.

Anschliessend werden durch Backpropagation die Gradienten bestimmt. Der AdamW-Optimizer verwendet diese Gradienten, um die Modellparameter zu aktualisieren.

Zur Überprüfung wird der Loss vor und nach einem Trainingsschritt auf demselben Batch verglichen.

In [21]:
# Modell in den Trainingsmodus versetzen
model.train()

# Ersten Batch auswählen
batch = next(iter(train_loader))

# Loss vor dem Trainingsschritt berechnen
logits = model(batch)
loss_before = criterion(logits, batch["label"])

# Alte Gradienten zurücksetzen
optimizer.zero_grad()

# Gradienten berechnen (Backpropagation)
loss_before.backward()

# Modellparameter aktualisieren
optimizer.step()

# Loss nach dem Trainingsschritt berechnen
with torch.no_grad():
    logits_after = model(batch)
    loss_after = criterion(logits_after, batch["label"])

print(f"Loss vorher: {loss_before.item():.4f}")
print(f"Loss nachher: {loss_after.item():.4f}")

Loss vorher: 0.6967
Loss nachher: 0.6907


### 2.9 Experiment Tracking mit Weights & Biases

Zur Dokumentation und Auswertung der Trainingsexperimente wird Weights & Biases (W&B) verwendet.

Für jedes Experiment werden die Modellkonfiguration sowie der Trainings- und Validierungsverlauf gespeichert. Dadurch lassen sich verschiedene Modelle und Hyperparameter vergleichen.

In [22]:
import os
import wandb

os.environ["WANDB_BASE_URL"] = "https://api.wandb.ai"
os.environ["WANDB_NOTEBOOK_NAME"] = "02_word_embeddings_model.ipynb"

wandb.require("core")

wandb: WARNING `wandb.require('core')` is a no-op as it is now the default behavior.


### 2.10 W&B-Run initialisieren

Für das Word-Embedding-Modell wird ein Experiment in Weights & Biases erstellt.

Die verwendeten Hyperparameter werden gespeichert, damit die Ergebnisse später nachvollzogen und mit dem RNN-Modell verglichen werden können.

In [23]:
run = wandb.init(
    project="NLP-Project-1",
    name="word-embeddings-baseline",
    config={
        "model": "WordEmbeddingModel",
        "embedding_dim": 50,
        "hidden_dim": 64,
        "batch_size": 32,
        "learning_rate": 0.001,
        "weight_decay": 0.01,
        "optimizer": "AdamW",
        "loss_function": "CrossEntropyLoss",
        "epochs": 10,
        "pretrained": False,
        "truncation": False
    }
)

print("W&B Run:", run.name)
print("W&B URL:", run.url)

W&B Run: word-embeddings-baseline
W&B URL: https://forge.coreweave.com/wandb/marco-blum-hochschule-luzern/NLP-Project-1/runs/oel5d008


### 2.11 Training vorbereiten

Vor dem vollständigen Training wird das Modell neu initialisiert, damit das Experiment mit untrainierten Gewichten beginnt.

Zur Reproduzierbarkeit wird ein fester Random Seed verwendet. Das Training erfolgt nach Möglichkeit auf der GPU.

In [24]:
# Reproduzierbarkeit
torch.manual_seed(42)

# GPU verwenden, falls verfügbar
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modell neu initialisieren
model = WordEmbeddingModel(
    vocab_size=len(vocab),
    embedding_dim=50
).to(device)

# Optimizer neu erstellen
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01
)

# Loss-Funktion
criterion = nn.CrossEntropyLoss()

print("Device:", device)
print("Modellparameter:", sum(p.numel() for p in model.parameters()))

Device: cuda
Modellparameter: 800629


### 2.12 Trainingsschleife

Die Trainingsfunktion verarbeitet sämtliche Batches des Trainingsdatensatzes.

Für jeden Batch werden die Vorhersagen berechnet, der Loss bestimmt und die Modellparameter mittels Backpropagation und AdamW aktualisiert.

Zusätzlich werden der durchschnittliche Trainings-Loss und die Trainings-Accuracy für eine Epoche berechnet.

In [25]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:

        # Daten auf die GPU verschieben
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Alte Gradienten zurücksetzen
        optimizer.zero_grad()

        # Vorhersagen berechnen
        logits = model(batch)

        # Loss berechnen
        loss = criterion(logits, batch["label"])

        # Backpropagation
        loss.backward()

        # Gewichte aktualisieren
        optimizer.step()

        # Statistiken sammeln
        batch_size = batch["label"].size(0)

        total_loss += loss.item() * batch_size

        predictions = torch.argmax(logits, dim=1)

        correct += (predictions == batch["label"]).sum().item()
        total += batch_size

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

### 2.13 Validierung

Nach jeder Trainingsepoche wird das Modell auf dem Validierungsdatensatz ausgewertet.

Dabei werden der durchschnittliche Validation Loss und die Validation Accuracy berechnet.

Während der Validierung werden keine Modellparameter aktualisiert. Dadurch lässt sich überprüfen, wie gut das Modell auf bisher ungesehenen Beispielen generalisiert.

In [26]:
def validate(model, valid_loader, criterion, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in valid_loader:

            # Daten auf die GPU verschieben
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            # Vorhersagen berechnen
            logits = model(batch)

            # Loss berechnen
            loss = criterion(logits, batch["label"])

            # Statistiken sammeln
            batch_size = batch["label"].size(0)

            total_loss += loss.item() * batch_size

            predictions = torch.argmax(logits, dim=1)

            correct += (predictions == batch["label"]).sum().item()
            total += batch_size

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

### 2.14 Training und Experiment Tracking

Das Word-Embedding-Modell wird über maximal 10 Epochen trainiert.

Nach jeder Epoche werden Trainings- und Validierungsmetriken berechnet und in Weights & Biases gespeichert.

Anhand dieser Metriken lässt sich untersuchen, ob das Modell lernt und wie gut es auf unbekannten Daten generalisiert.

In [27]:
epochs = 10

for epoch in range(epochs):

    # Eine Trainingsepoche durchführen
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    # Modell auf Validierungsdaten auswerten
    valid_loss, valid_acc = validate(
        model, valid_loader, criterion, device
    )

    # Ergebnisse in W&B speichern
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_acc
    })

    # Ergebnisse ausgeben
    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2%} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.2%}"
    )

Epoch 1/10 | Train Loss: 0.6759 | Train Acc: 57.25% | Valid Loss: 0.6734 | Valid Acc: 58.70%
Epoch 2/10 | Train Loss: 0.6285 | Train Acc: 63.87% | Valid Loss: 0.6534 | Valid Acc: 60.40%
Epoch 3/10 | Train Loss: 0.5748 | Train Acc: 69.64% | Valid Loss: 0.6576 | Valid Acc: 61.30%
Epoch 4/10 | Train Loss: 0.5221 | Train Acc: 73.52% | Valid Loss: 0.6592 | Valid Acc: 63.50%
Epoch 5/10 | Train Loss: 0.4714 | Train Acc: 76.77% | Valid Loss: 0.6816 | Valid Acc: 64.20%
Epoch 6/10 | Train Loss: 0.4228 | Train Acc: 80.14% | Valid Loss: 0.7108 | Valid Acc: 64.40%
Epoch 7/10 | Train Loss: 0.3760 | Train Acc: 83.11% | Valid Loss: 0.7508 | Valid Acc: 64.20%
Epoch 8/10 | Train Loss: 0.3323 | Train Acc: 85.24% | Valid Loss: 0.7807 | Valid Acc: 63.30%
Epoch 9/10 | Train Loss: 0.2902 | Train Acc: 87.64% | Valid Loss: 0.8327 | Valid Acc: 63.70%
Epoch 10/10 | Train Loss: 0.2504 | Train Acc: 89.66% | Valid Loss: 0.8859 | Valid Acc: 64.30%


### 2.15 Model Checkpointing

Während des Trainings wird das Modell gespeichert, sobald eine höhere Validation Accuracy erreicht wird.

Dadurch bleibt der beste Modellzustand erhalten, auch wenn sich die Validierungsleistung in späteren Epochen verschlechtert.

Die gespeicherten Gewichte können anschliessend für die Evaluation verwendet werden.

In [ ]:
from pathlib import Path

# Speicherort für Modellgewichte
checkpoint_dir = project_root / "outputs" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / (
    f"word_embeddings_bs{batch_size}_"
    f"lr{learning_rate:g}_ep{epochs}_p{patience}_s{seed}.pt"
)

epochs = 10
best_valid_acc = 0.0
best_epoch = 0

for epoch in range(epochs):

    # Training
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    # Validierung
    valid_loss, valid_acc = validate(
        model, valid_loader, criterion, device
    )

    # Bestes Modell speichern
    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_epoch = epoch + 1

        torch.save({
            "epoch": best_epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "valid_accuracy": best_valid_acc,
            "vocab": vocab
        }, checkpoint_path)

    # Metriken an W&B senden
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_acc
    })

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2%} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.2%}"
    )

print(f"\nBeste Epoche: {best_epoch}")
print(f"Beste Validation Accuracy: {best_valid_acc:.2%}")
print(f"Checkpoint gespeichert unter: {checkpoint_path}")

Epoch 1/10 | Train Loss: 0.2154 | Train Acc: 91.38% | Valid Loss: 0.9358 | Valid Acc: 64.10%
Epoch 2/10 | Train Loss: 0.1817 | Train Acc: 93.36% | Valid Loss: 0.9918 | Valid Acc: 65.10%
Epoch 3/10 | Train Loss: 0.1522 | Train Acc: 94.70% | Valid Loss: 1.0588 | Valid Acc: 64.30%
Epoch 4/10 | Train Loss: 0.1258 | Train Acc: 96.07% | Valid Loss: 1.1295 | Valid Acc: 64.60%
Epoch 5/10 | Train Loss: 0.1036 | Train Acc: 96.95% | Valid Loss: 1.1948 | Valid Acc: 64.20%
Epoch 6/10 | Train Loss: 0.0847 | Train Acc: 97.59% | Valid Loss: 1.2788 | Valid Acc: 63.70%
Epoch 7/10 | Train Loss: 0.0692 | Train Acc: 98.21% | Valid Loss: 1.3557 | Valid Acc: 63.60%
Epoch 8/10 | Train Loss: 0.0568 | Train Acc: 98.55% | Valid Loss: 1.4307 | Valid Acc: 63.40%


Failed to export metrics batch due to timeout, max retries or shutdown.


Epoch 9/10 | Train Loss: 0.0460 | Train Acc: 98.84% | Valid Loss: 1.5192 | Valid Acc: 62.90%
Epoch 10/10 | Train Loss: 0.0380 | Train Acc: 99.03% | Valid Loss: 1.6023 | Valid Acc: 63.90%

Beste Epoche: 2
Beste Validation Accuracy: 65.10%
Checkpoint gespeichert unter: C:\Users\Marco\Marco\Github\NLP-Project-1\outputs\checkpoints\word_embeddings_best.pt


### 2.16 Neues Baseline-Experiment

Für ein reproduzierbares Experiment werden das Modell und der Optimizer neu initialisiert.

Das Experiment verwendet dieselbe Architektur und dieselben Hyperparameter wie die bisherige Baseline. Zusätzlich werden Model Checkpointing und Early Stopping eingesetzt, um den besten Modellzustand zu sichern und unnötiges Overfitting zu begrenzen.

In [29]:
# Zufallszahlen für reproduzierbare Modellinitialisierung
torch.manual_seed(42)

# Neues, untrainiertes Modell erstellen
model = WordEmbeddingModel(
    vocab_size=len(vocab),
    embedding_dim=50
).to(device)

# Neuen Optimizer erstellen
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01
)

print("Neues Modell initialisiert.")
print("Device:", device)

Neues Modell initialisiert.
Device: cuda


### 2.17 Neues W&B-Experiment

Für das erneute Training wird ein separater W&B-Run erstellt.

Die Modellarchitektur und Hyperparameter bleiben gegenüber der bisherigen Baseline unverändert. Zusätzlich werden Checkpointing und Early Stopping eingesetzt.

In [30]:
# Bisherigen W&B-Run abschliessen
wandb.finish()

# Neuen Experiment-Run starten
run = wandb.init(
    project="NLP-Project-1",
    name="WE | Baseline | Checkpoint + EarlyStopping",
    config={
        "model": "WordEmbeddingModel",
        "embedding_dim": 50,
        "hidden_dim": 64,
        "batch_size": 32,
        "learning_rate": 0.001,
        "weight_decay": 0.01,
        "optimizer": "AdamW",
        "epochs": 10,
        "seed": 42,
        "checkpointing": True,
        "early_stopping": True,
        "pretrained": False,
        "truncation": False
    }
)

print("Neuer W&B-Run:", run.name)
print("Run-URL:", run.url)

epoch,▁▂▃▃▄▅▆▆▇█▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▂▃▄▄▅▅▆▆▆▇▇▇███████
train_loss,█▇▇▆▆▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁
valid_accuracy,▁▃▄▆▇▇▇▆▆▇▇█▇▇▇▆▆▆▆▇
valid_loss,▁▁▁▁▁▁▂▂▂▃▃▃▄▅▅▆▆▇▇█
epoch,10
train_accuracy,0.9903
train_loss,0.03803
valid_accuracy,0.639
valid_loss,1.60232


Neuer W&B-Run: WE | Baseline | Checkpoint + EarlyStopping
Run-URL: https://forge.coreweave.com/wandb/marco-blum-hochschule-luzern/NLP-Project-1/runs/897oenwj


### 2.18 Early Stopping

Early Stopping beendet das Training vorzeitig, wenn sich die Validation Accuracy über mehrere Epochen nicht verbessert.

Mit einer Patience von 3 werden maximal drei aufeinanderfolgende Epochen ohne Verbesserung toleriert.

Zusätzlich wird der Modellzustand mit der höchsten Validation Accuracy gespeichert.

In [31]:
from pathlib import Path

# Speicherort für das beste Modell
checkpoint_dir = project_root / "outputs" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "word_embeddings_early_stopping.pt"

epochs = 10
patience = 3

best_valid_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(epochs):

    # Training
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    # Validierung
    valid_loss, valid_acc = validate(
        model, valid_loader, criterion, device
    )

    # Prüfen, ob sich die Validation Accuracy verbessert hat
    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        # Bestes Modell speichern
        torch.save({
            "epoch": best_epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "valid_accuracy": best_valid_acc,
            "vocab": vocab
        }, checkpoint_path)

    else:
        epochs_without_improvement += 1

    # Metriken in W&B speichern
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_acc
    })

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2%} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.2%}"
    )

    # Early Stopping
    if epochs_without_improvement >= patience:
        print(f"\nEarly Stopping nach Epoche {epoch + 1}")
        break

print(f"\nBeste Epoche: {best_epoch}")
print(f"Beste Validation Accuracy: {best_valid_acc:.2%}")
print(f"Checkpoint: {checkpoint_path}")

Epoch 1/10 | Train Loss: 0.6759 | Train Acc: 57.25% | Valid Loss: 0.6734 | Valid Acc: 58.70%
Epoch 2/10 | Train Loss: 0.6285 | Train Acc: 63.87% | Valid Loss: 0.6534 | Valid Acc: 60.40%
Epoch 3/10 | Train Loss: 0.5748 | Train Acc: 69.64% | Valid Loss: 0.6576 | Valid Acc: 61.30%
Epoch 4/10 | Train Loss: 0.5221 | Train Acc: 73.52% | Valid Loss: 0.6592 | Valid Acc: 63.50%
Epoch 5/10 | Train Loss: 0.4714 | Train Acc: 76.77% | Valid Loss: 0.6816 | Valid Acc: 64.20%
Epoch 6/10 | Train Loss: 0.4228 | Train Acc: 80.14% | Valid Loss: 0.7108 | Valid Acc: 64.40%
Epoch 7/10 | Train Loss: 0.3760 | Train Acc: 83.11% | Valid Loss: 0.7508 | Valid Acc: 64.20%
Epoch 8/10 | Train Loss: 0.3323 | Train Acc: 85.24% | Valid Loss: 0.7807 | Valid Acc: 63.30%
Epoch 9/10 | Train Loss: 0.2902 | Train Acc: 87.64% | Valid Loss: 0.8327 | Valid Acc: 63.70%

Early Stopping nach Epoche 9

Beste Epoche: 6
Beste Validation Accuracy: 64.40%
Checkpoint: C:\Users\Marco\Marco\Github\NLP-Project-1\outputs\checkpoints\word_emb

Failed to export metrics batch due to timeout, max retries or shutdown.


### 2.19 GPU-Speicher und Batch Size

Zur Bestimmung der maximal möglichen Batch Size wird der verfügbare GPU-Speicher untersucht.

Anschliessend werden unterschiedliche Batchgrössen getestet, um festzustellen, wie viele Trainingsbeispiele gleichzeitig verarbeitet werden können, ohne den GPU-Speicher zu überschreiten.

In [32]:
# GPU-Speicher überprüfen
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)

    total_memory = gpu.total_memory / (1024 ** 3)
    allocated_memory = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved_memory = torch.cuda.memory_reserved() / (1024 ** 3)

    print("GPU:", gpu.name)
    print(f"Gesamter GPU-Speicher: {total_memory:.2f} GB")
    print(f"Von PyTorch belegt: {allocated_memory:.2f} GB")
    print(f"Von PyTorch reserviert: {reserved_memory:.2f} GB")

GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Gesamter GPU-Speicher: 6.00 GB
Von PyTorch belegt: 0.03 GB
Von PyTorch reserviert: 0.04 GB


### 2.20 Maximale Batch Size bestimmen

Die Batch Size wird schrittweise erhöht, um den GPU-Speicherbedarf während eines vollständigen Trainingsschritts zu untersuchen.

Dabei werden Forward Pass, Loss-Berechnung, Backpropagation und Optimizer-Schritt ausgeführt.

Der Test erfolgt mit einem separaten Modell, damit die bisherigen Trainingsergebnisse unverändert bleiben.

In [33]:
import gc
import time

# Batchgrössen, die getestet werden
batch_sizes = [32, 64, 128, 256, 512, 1024, 2048, 4096]

results = []

for bs in batch_sizes:

    # Separates Modell für den Test
    test_model = WordEmbeddingModel(
        vocab_size=len(vocab),
        embedding_dim=50
    ).to(device)

    test_optimizer = torch.optim.AdamW(
        test_model.parameters(),
        lr=0.001,
        weight_decay=0.01
    )

    # Batch mit der gewünschten Grösse erstellen
    test_loader = DataLoader(
        train_encoded,
        batch_size=bs,
        shuffle=False,
        collate_fn=collate_batch
    )

    batch = next(iter(test_loader))
    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()

    try:
        test_model.train()
        test_optimizer.zero_grad()

        logits = test_model(batch)
        loss = criterion(logits, batch["label"])

        loss.backward()
        test_optimizer.step()

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

        print(
            f"Batch Size {bs:4d} | "
            f"GPU Peak: {peak_memory:.2f} GB | "
            f"Zeit: {elapsed:.3f} s"
        )

        results.append((bs, peak_memory, elapsed))

    except torch.OutOfMemoryError:
        print(f"Batch Size {bs}: CUDA Out of Memory")
        break

    finally:
        del test_model, test_optimizer, test_loader, batch
        gc.collect()
        torch.cuda.empty_cache()

Batch Size   32 | GPU Peak: 0.04 GB | Zeit: 0.012 s
Batch Size   64 | GPU Peak: 0.04 GB | Zeit: 0.011 s
Batch Size  128 | GPU Peak: 0.04 GB | Zeit: 0.058 s
Batch Size  256 | GPU Peak: 0.04 GB | Zeit: 0.021 s
Batch Size  512 | GPU Peak: 0.04 GB | Zeit: 0.014 s
Batch Size 1024 | GPU Peak: 0.05 GB | Zeit: 0.017 s
Batch Size 2048 | GPU Peak: 0.07 GB | Zeit: 0.035 s
Batch Size 4096 | GPU Peak: 0.10 GB | Zeit: 0.123 s


In [34]:
batch_sizes = [8192, 16384, 16801]

for bs in batch_sizes:

    test_model = WordEmbeddingModel(
        vocab_size=len(vocab),
        embedding_dim=50
    ).to(device)

    test_optimizer = torch.optim.AdamW(
        test_model.parameters(),
        lr=0.001,
        weight_decay=0.01
    )

    test_loader = DataLoader(
        train_encoded,
        batch_size=bs,
        shuffle=False,
        collate_fn=collate_batch
    )

    batch = next(iter(test_loader))
    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    try:
        torch.cuda.reset_peak_memory_stats()
        start = time.perf_counter()

        test_model.train()
        test_optimizer.zero_grad()

        logits = test_model(batch)
        loss = criterion(logits, batch["label"])

        loss.backward()
        test_optimizer.step()

        torch.cuda.synchronize()

        elapsed = time.perf_counter() - start
        peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

        print(
            f"Batch Size {bs:5d} | "
            f"GPU Peak: {peak_memory:.2f} GB | "
            f"Zeit: {elapsed:.3f} s"
        )

    except torch.OutOfMemoryError:
        print(f"Batch Size {bs}: CUDA Out of Memory")
        break

    finally:
        del test_model, test_optimizer, test_loader, batch
        gc.collect()
        torch.cuda.empty_cache()

Batch Size  8192 | GPU Peak: 0.16 GB | Zeit: 0.293 s
Batch Size 16384 | GPU Peak: 0.29 GB | Zeit: 0.607 s
Batch Size 16801 | GPU Peak: 0.29 GB | Zeit: 0.328 s


### 2.21 Systematische Hyperparameteroptimierung

Zur Untersuchung des Einflusses der Batch Size wird eine wiederverwendbare Experimentfunktion implementiert.

Für jedes Experiment werden das Modell und der Optimizer neu initialisiert. Die übrigen Hyperparameter bleiben zunächst unverändert, sodass die Ergebnisse der verschiedenen Batchgrössen verglichen werden können.

Die Trainings- und Validierungsmetriken werden für jedes Experiment separat in Weights & Biases (W&B) protokolliert. Das Modell mit der höchsten Validation Accuracy wird als Checkpoint gespeichert. Early Stopping beendet das Training, wenn sich die Validation Accuracy während drei aufeinanderfolgenden Epochen nicht verbessert.

In [ ]:
import importlib
import src.models
import src.training

importlib.reload(src.models)
importlib.reload(src.training)


def run_experiment(
    batch_size,
    learning_rate=0.001,
    epochs=60,
    patience=5,
    seed=42,
    dropout=0.0,
    weight_decay=0.01
):
    return src.training.run_experiment(
        train_encoded=train_encoded,
        valid_encoded=valid_encoded,
        vocab=vocab,
        project_root=project_root,
        device=device,
        batch_size=batch_size,
        learning_rate=learning_rate,
        epochs=epochs,
        patience=patience,
        seed=seed,
        dropout=dropout,
        weight_decay=weight_decay
    )

In [36]:
result_bs16 = run_experiment(batch_size=16)

epoch,▁▂▃▄▅▅▆▇█
train_accuracy,▁▃▄▅▅▆▇▇█
train_loss,█▇▆▅▄▃▃▂▁
valid_accuracy,▁▃▄▇███▇▇
valid_loss,▂▁▁▁▂▃▅▆█
epoch,9
train_accuracy,0.87638
train_loss,0.29015
valid_accuracy,0.637
valid_loss,0.83272


BS 16 | Epoch 1/10 | Train Acc: 58.21% | Valid Acc: 58.10%
BS 16 | Epoch 2/10 | Train Acc: 65.98% | Valid Acc: 62.20%
BS 16 | Epoch 3/10 | Train Acc: 71.81% | Valid Acc: 63.70%
BS 16 | Epoch 4/10 | Train Acc: 76.10% | Valid Acc: 65.40%
BS 16 | Epoch 5/10 | Train Acc: 80.06% | Valid Acc: 64.90%
BS 16 | Epoch 6/10 | Train Acc: 83.01% | Valid Acc: 66.30%
BS 16 | Epoch 7/10 | Train Acc: 86.04% | Valid Acc: 65.40%
BS 16 | Epoch 8/10 | Train Acc: 88.99% | Valid Acc: 65.80%
BS 16 | Epoch 9/10 | Train Acc: 91.30% | Valid Acc: 65.60%
Early Stopping nach Epoche 9

Beste Validation Accuracy: 66.30%
Beste Epoche: 6
Trainingszeit: 47.4 Sekunden


epoch,▁▂▃▄▅▅▆▇█
train_accuracy,▁▃▄▅▆▆▇██
train_loss,█▇▆▅▄▃▃▂▁
valid_accuracy,▁▅▆▇▇█▇█▇
valid_loss,▁▁▁▁▂▃▅▆█
best_epoch,6
best_valid_accuracy,0.663
epoch,9
train_accuracy,0.91304
train_loss,0.21871
training_time_seconds,47.43642


In [37]:
result_bs64 = run_experiment(batch_size=64)

BS 64 | Epoch 1/10 | Train Acc: 56.45% | Valid Acc: 55.90%
BS 64 | Epoch 2/10 | Train Acc: 62.41% | Valid Acc: 59.10%
BS 64 | Epoch 3/10 | Train Acc: 66.82% | Valid Acc: 60.00%
BS 64 | Epoch 4/10 | Train Acc: 70.53% | Valid Acc: 60.80%
BS 64 | Epoch 5/10 | Train Acc: 74.35% | Valid Acc: 62.50%
BS 64 | Epoch 6/10 | Train Acc: 77.21% | Valid Acc: 62.70%
BS 64 | Epoch 7/10 | Train Acc: 80.04% | Valid Acc: 63.40%
BS 64 | Epoch 8/10 | Train Acc: 82.55% | Valid Acc: 64.00%
BS 64 | Epoch 9/10 | Train Acc: 84.72% | Valid Acc: 64.70%
BS 64 | Epoch 10/10 | Train Acc: 86.52% | Valid Acc: 64.30%

Beste Validation Accuracy: 64.70%
Beste Epoche: 9
Trainingszeit: 22.7 Sekunden


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▂▃▄▅▆▆▇██
train_loss,█▇▇▆▅▄▃▂▂▁
valid_accuracy,▁▄▄▅▆▆▇▇██
valid_loss,▂▁▁▁▁▂▄▄▆█
best_epoch,9
best_valid_accuracy,0.647
epoch,10
train_accuracy,0.86519
train_loss,0.31447
training_time_seconds,22.70583


In [38]:
result_bs128 = run_experiment(batch_size=128)

BS 128 | Epoch 1/10 | Train Acc: 55.29% | Valid Acc: 54.70%
BS 128 | Epoch 2/10 | Train Acc: 60.25% | Valid Acc: 58.80%
BS 128 | Epoch 3/10 | Train Acc: 64.48% | Valid Acc: 59.90%
BS 128 | Epoch 4/10 | Train Acc: 67.64% | Valid Acc: 59.70%
BS 128 | Epoch 5/10 | Train Acc: 70.89% | Valid Acc: 61.30%
BS 128 | Epoch 6/10 | Train Acc: 73.73% | Valid Acc: 62.40%
BS 128 | Epoch 7/10 | Train Acc: 76.10% | Valid Acc: 63.20%
BS 128 | Epoch 8/10 | Train Acc: 78.35% | Valid Acc: 64.40%
BS 128 | Epoch 9/10 | Train Acc: 80.35% | Valid Acc: 64.80%
BS 128 | Epoch 10/10 | Train Acc: 82.31% | Valid Acc: 64.80%

Beste Validation Accuracy: 64.80%
Beste Epoche: 9
Trainingszeit: 14.6 Sekunden


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▂▃▄▅▆▆▇▇█
train_loss,█▇▇▆▅▄▃▂▂▁
valid_accuracy,▁▄▅▄▆▆▇███
valid_loss,▄▃▂▁▁▂▃▄▆█
best_epoch,9
best_valid_accuracy,0.648
epoch,10
train_accuracy,0.82311
train_loss,0.38879
training_time_seconds,14.61012


### 2.23 Untersuchung der Trainingsdauer

Nach dem Vergleich verschiedener Batchgrössen wird die benötigte Trainingsdauer untersucht.

Die maximale Anzahl Epochen wird auf 30 erhöht. Early Stopping mit einer Patience von drei Epochen bleibt aktiviert, um das Training bei ausbleibender Verbesserung der Validation Accuracy vorzeitig zu beenden.

Da grössere Batchgrössen weniger Gewichtsaktualisierungen pro Epoche durchführen, wird zusätzlich überprüft, ob diese von einer längeren Trainingsdauer profitieren.

In [40]:
result_bs128_30 = run_experiment(
    batch_size=128,
    epochs=30,
    patience=5
)

BS 128 | Epoch 1/30 | Train Acc: 55.29% | Valid Acc: 54.70%
BS 128 | Epoch 2/30 | Train Acc: 60.25% | Valid Acc: 58.80%
BS 128 | Epoch 3/30 | Train Acc: 64.48% | Valid Acc: 59.90%
BS 128 | Epoch 4/30 | Train Acc: 67.64% | Valid Acc: 59.70%
BS 128 | Epoch 5/30 | Train Acc: 70.89% | Valid Acc: 61.30%
BS 128 | Epoch 6/30 | Train Acc: 73.73% | Valid Acc: 62.40%
BS 128 | Epoch 7/30 | Train Acc: 76.10% | Valid Acc: 63.20%
BS 128 | Epoch 8/30 | Train Acc: 78.35% | Valid Acc: 64.40%
BS 128 | Epoch 9/30 | Train Acc: 80.35% | Valid Acc: 64.80%
BS 128 | Epoch 10/30 | Train Acc: 82.31% | Valid Acc: 64.80%
BS 128 | Epoch 11/30 | Train Acc: 84.08% | Valid Acc: 65.70%
BS 128 | Epoch 12/30 | Train Acc: 85.77% | Valid Acc: 65.50%
BS 128 | Epoch 13/30 | Train Acc: 87.23% | Valid Acc: 64.60%
BS 128 | Epoch 14/30 | Train Acc: 88.59% | Valid Acc: 66.00%
BS 128 | Epoch 15/30 | Train Acc: 89.86% | Valid Acc: 66.10%
BS 128 | Epoch 16/30 | Train Acc: 91.04% | Valid Acc: 65.50%
BS 128 | Epoch 17/30 | Train Acc:

epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
train_accuracy,▁▂▃▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇████
train_loss,██▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁
valid_accuracy,▁▃▄▄▅▆▆▇▇▇▇▇▇██▇█▇██▇▇
valid_loss,▁▁▁▁▁▁▁▁▂▂▂▃▃▄▄▄▅▆▆▇▇█
best_epoch,17
best_valid_accuracy,0.666
epoch,22
train_accuracy,0.96078
train_loss,0.12686
training_time_seconds,30.92059


In [41]:
result_bs64_30 = run_experiment(
    batch_size=64,
    epochs=30,
    patience=5
)

BS 64 | Epoch 1/30 | Train Acc: 56.45% | Valid Acc: 55.90%
BS 64 | Epoch 2/30 | Train Acc: 62.41% | Valid Acc: 59.10%
BS 64 | Epoch 3/30 | Train Acc: 66.82% | Valid Acc: 60.00%
BS 64 | Epoch 4/30 | Train Acc: 70.53% | Valid Acc: 60.80%
BS 64 | Epoch 5/30 | Train Acc: 74.35% | Valid Acc: 62.50%
BS 64 | Epoch 6/30 | Train Acc: 77.21% | Valid Acc: 62.70%
BS 64 | Epoch 7/30 | Train Acc: 80.04% | Valid Acc: 63.40%
BS 64 | Epoch 8/30 | Train Acc: 82.55% | Valid Acc: 64.00%
BS 64 | Epoch 9/30 | Train Acc: 84.72% | Valid Acc: 64.70%
BS 64 | Epoch 10/30 | Train Acc: 86.52% | Valid Acc: 64.30%
BS 64 | Epoch 11/30 | Train Acc: 88.38% | Valid Acc: 64.40%
BS 64 | Epoch 12/30 | Train Acc: 90.15% | Valid Acc: 64.90%
BS 64 | Epoch 13/30 | Train Acc: 91.79% | Valid Acc: 64.80%
BS 64 | Epoch 14/30 | Train Acc: 93.14% | Valid Acc: 64.20%
BS 64 | Epoch 15/30 | Train Acc: 94.30% | Valid Acc: 63.90%
BS 64 | Epoch 16/30 | Train Acc: 95.29% | Valid Acc: 63.70%
BS 64 | Epoch 17/30 | Train Acc: 96.25% | Valid A

epoch,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
train_accuracy,▁▂▃▃▄▅▅▆▆▆▇▇▇▇███
train_loss,██▇▆▆▅▅▄▄▃▃▃▂▂▂▁▁
valid_accuracy,▁▃▄▅▆▆▇▇█████▇▇▇█
valid_loss,▁▁▁▁▁▁▂▂▃▃▄▄▅▆▆▇█
best_epoch,12
best_valid_accuracy,0.649
epoch,17
train_accuracy,0.9625
train_loss,0.12577
training_time_seconds,32.91549


In [42]:
result_bs16_30 = run_experiment(
    batch_size=16,
    epochs=30,
    patience=5
)

BS 16 | Epoch 1/30 | Train Acc: 58.21% | Valid Acc: 58.10%
BS 16 | Epoch 2/30 | Train Acc: 65.98% | Valid Acc: 62.20%
BS 16 | Epoch 3/30 | Train Acc: 71.81% | Valid Acc: 63.70%
BS 16 | Epoch 4/30 | Train Acc: 76.10% | Valid Acc: 65.40%
BS 16 | Epoch 5/30 | Train Acc: 80.06% | Valid Acc: 64.90%
BS 16 | Epoch 6/30 | Train Acc: 83.01% | Valid Acc: 66.30%
BS 16 | Epoch 7/30 | Train Acc: 86.04% | Valid Acc: 65.40%
BS 16 | Epoch 8/30 | Train Acc: 88.99% | Valid Acc: 65.80%
BS 16 | Epoch 9/30 | Train Acc: 91.30% | Valid Acc: 65.60%
BS 16 | Epoch 10/30 | Train Acc: 93.39% | Valid Acc: 64.40%
BS 16 | Epoch 11/30 | Train Acc: 95.42% | Valid Acc: 64.10%
Early Stopping nach Epoche 11

Beste Validation Accuracy: 66.30%
Beste Epoche: 6
Trainingszeit: 65.2 Sekunden


epoch,▁▂▂▃▄▅▅▆▇▇█
train_accuracy,▁▂▄▄▅▆▆▇▇██
train_loss,█▇▆▆▅▄▃▃▂▂▁
valid_accuracy,▁▅▆▇▇█▇█▇▆▆
valid_loss,▁▁▁▁▂▂▃▄▆▆█
best_epoch,6
best_valid_accuracy,0.663
epoch,11
train_accuracy,0.95423
train_loss,0.13802
training_time_seconds,65.2076


In [43]:
result_bs32_30 = run_experiment(
    batch_size=32,
    epochs=30,
    patience=5
)

BS 32 | Epoch 1/30 | Train Acc: 57.14% | Valid Acc: 57.90%
BS 32 | Epoch 2/30 | Train Acc: 64.22% | Valid Acc: 60.00%
BS 32 | Epoch 3/30 | Train Acc: 69.42% | Valid Acc: 61.60%
BS 32 | Epoch 4/30 | Train Acc: 73.75% | Valid Acc: 64.50%
BS 32 | Epoch 5/30 | Train Acc: 77.28% | Valid Acc: 63.50%
BS 32 | Epoch 6/30 | Train Acc: 80.02% | Valid Acc: 64.70%
BS 32 | Epoch 7/30 | Train Acc: 83.07% | Valid Acc: 65.00%
BS 32 | Epoch 8/30 | Train Acc: 85.58% | Valid Acc: 66.20%
BS 32 | Epoch 9/30 | Train Acc: 87.39% | Valid Acc: 65.50%
BS 32 | Epoch 10/30 | Train Acc: 89.73% | Valid Acc: 65.00%
BS 32 | Epoch 11/30 | Train Acc: 91.96% | Valid Acc: 65.30%
BS 32 | Epoch 12/30 | Train Acc: 93.57% | Valid Acc: 65.80%
BS 32 | Epoch 13/30 | Train Acc: 94.85% | Valid Acc: 63.90%
Early Stopping nach Epoche 13

Beste Validation Accuracy: 66.20%
Beste Epoche: 8
Trainingszeit: 43.8 Sekunden


epoch,▁▂▂▃▃▄▅▅▆▆▇▇█
train_accuracy,▁▂▃▄▅▅▆▆▇▇▇██
train_loss,█▇▇▆▅▅▄▃▃▂▂▁▁
valid_accuracy,▁▃▄▇▆▇▇█▇▇▇█▆
valid_loss,▁▁▁▁▁▂▃▃▄▅▆▇█
best_epoch,8
best_valid_accuracy,0.662
epoch,13
train_accuracy,0.94851
train_loss,0.15138
training_time_seconds,43.82259


In [44]:
result_bs256_30 = run_experiment(
    batch_size=256,
    epochs=30,
    patience=5
)

BS 256 | Epoch 1/30 | Train Acc: 54.51% | Valid Acc: 53.20%
BS 256 | Epoch 2/30 | Train Acc: 58.73% | Valid Acc: 57.00%
BS 256 | Epoch 3/30 | Train Acc: 61.97% | Valid Acc: 57.60%
BS 256 | Epoch 4/30 | Train Acc: 64.59% | Valid Acc: 59.70%
BS 256 | Epoch 5/30 | Train Acc: 67.33% | Valid Acc: 60.10%
BS 256 | Epoch 6/30 | Train Acc: 69.84% | Valid Acc: 60.50%
BS 256 | Epoch 7/30 | Train Acc: 71.91% | Valid Acc: 61.80%
BS 256 | Epoch 8/30 | Train Acc: 73.95% | Valid Acc: 62.70%
BS 256 | Epoch 9/30 | Train Acc: 75.79% | Valid Acc: 63.50%
BS 256 | Epoch 10/30 | Train Acc: 77.65% | Valid Acc: 64.60%
BS 256 | Epoch 11/30 | Train Acc: 79.13% | Valid Acc: 64.30%
BS 256 | Epoch 12/30 | Train Acc: 80.76% | Valid Acc: 65.40%
BS 256 | Epoch 13/30 | Train Acc: 82.24% | Valid Acc: 65.00%
BS 256 | Epoch 14/30 | Train Acc: 83.55% | Valid Acc: 65.00%
BS 256 | Epoch 15/30 | Train Acc: 84.63% | Valid Acc: 64.60%
BS 256 | Epoch 16/30 | Train Acc: 85.86% | Valid Acc: 65.30%
BS 256 | Epoch 17/30 | Train Acc:

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_accuracy,▁▂▂▃▃▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇██████
train_loss,███▇▇▇▆▆▆▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
valid_accuracy,▁▃▃▄▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████▇
valid_loss,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▅▅▅▆▆▇▇██
best_epoch,28
best_valid_accuracy,0.669
epoch,30
train_accuracy,0.96119
train_loss,0.12683
training_time_seconds,35.39703


In [45]:
result_bs256_60 = run_experiment(
    batch_size=256,
    epochs=60,
    patience=5
)

BS 256 | Epoch 1/60 | Train Acc: 54.51% | Valid Acc: 53.20%
BS 256 | Epoch 2/60 | Train Acc: 58.73% | Valid Acc: 57.00%
BS 256 | Epoch 3/60 | Train Acc: 61.97% | Valid Acc: 57.60%
BS 256 | Epoch 4/60 | Train Acc: 64.59% | Valid Acc: 59.70%
BS 256 | Epoch 5/60 | Train Acc: 67.33% | Valid Acc: 60.10%
BS 256 | Epoch 6/60 | Train Acc: 69.84% | Valid Acc: 60.50%
BS 256 | Epoch 7/60 | Train Acc: 71.91% | Valid Acc: 61.80%
BS 256 | Epoch 8/60 | Train Acc: 73.95% | Valid Acc: 62.70%
BS 256 | Epoch 9/60 | Train Acc: 75.79% | Valid Acc: 63.50%
BS 256 | Epoch 10/60 | Train Acc: 77.65% | Valid Acc: 64.60%
BS 256 | Epoch 11/60 | Train Acc: 79.13% | Valid Acc: 64.30%
BS 256 | Epoch 12/60 | Train Acc: 80.76% | Valid Acc: 65.40%
BS 256 | Epoch 13/60 | Train Acc: 82.24% | Valid Acc: 65.00%
BS 256 | Epoch 14/60 | Train Acc: 83.55% | Valid Acc: 65.00%
BS 256 | Epoch 15/60 | Train Acc: 84.63% | Valid Acc: 64.60%
BS 256 | Epoch 16/60 | Train Acc: 85.86% | Valid Acc: 65.30%
BS 256 | Epoch 17/60 | Train Acc:

epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
train_accuracy,▁▂▂▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇████████
train_loss,███▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
valid_accuracy,▁▃▃▄▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████▇██▇
valid_loss,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▇▇██
best_epoch,28
best_valid_accuracy,0.669
epoch,33
train_accuracy,0.97149
train_loss,0.10268
training_time_seconds,34.57936


In [46]:
result_bs512_60 = run_experiment(
    batch_size=512,
    epochs=60,
    patience=5
)

BS 512 | Epoch 1/60 | Train Acc: 54.12% | Valid Acc: 52.60%
BS 512 | Epoch 2/60 | Train Acc: 57.31% | Valid Acc: 54.00%
BS 512 | Epoch 3/60 | Train Acc: 59.41% | Valid Acc: 56.30%
BS 512 | Epoch 4/60 | Train Acc: 61.94% | Valid Acc: 57.10%
BS 512 | Epoch 5/60 | Train Acc: 63.92% | Valid Acc: 57.60%
BS 512 | Epoch 6/60 | Train Acc: 65.69% | Valid Acc: 59.30%
BS 512 | Epoch 7/60 | Train Acc: 67.62% | Valid Acc: 59.40%
BS 512 | Epoch 8/60 | Train Acc: 69.45% | Valid Acc: 59.50%
BS 512 | Epoch 9/60 | Train Acc: 71.38% | Valid Acc: 61.90%
BS 512 | Epoch 10/60 | Train Acc: 73.14% | Valid Acc: 61.90%
BS 512 | Epoch 11/60 | Train Acc: 74.32% | Valid Acc: 61.70%
BS 512 | Epoch 12/60 | Train Acc: 75.75% | Valid Acc: 62.40%
BS 512 | Epoch 13/60 | Train Acc: 76.87% | Valid Acc: 63.40%
BS 512 | Epoch 14/60 | Train Acc: 78.17% | Valid Acc: 63.50%
BS 512 | Epoch 15/60 | Train Acc: 79.34% | Valid Acc: 64.10%
BS 512 | Epoch 16/60 | Train Acc: 80.45% | Valid Acc: 64.80%
BS 512 | Epoch 17/60 | Train Acc:

epoch,▁▁▁▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇██
train_accuracy,▁▂▂▂▃▃▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█████
train_loss,███▇▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁
valid_accuracy,▁▂▃▃▄▅▅▅▆▆▆▆▇▇▇▇█▇█▇██▇████▇█
valid_loss,▂▂▂▁▁▁▁▁▁▁▁▁▂▂▂▂▃▃▃▄▄▅▅▅▆▆▇██
best_epoch,24
best_valid_accuracy,0.658
epoch,29
train_accuracy,0.90637
train_loss,0.23802
training_time_seconds,24.31958


### 2.24 Ergebnisse der Batch-Size-Untersuchung

Zur Untersuchung des Einflusses der Batch Size wurden sechs unterschiedliche Konfigurationen (16, 32, 64, 128, 256 und 512) miteinander verglichen.

Zunächst wurde überprüft, welche Batchgrössen vom verfügbaren GPU-Speicher unterstützt werden. Dabei zeigte sich, dass selbst der gesamte Trainingsdatensatz mit 16'801 Beispielen gleichzeitig verarbeitet werden konnte. Der GPU-Speicher stellte somit für das Word-Embedding-Modell keine wesentliche Einschränkung dar.

Anschliessend wurden die verschiedenen Batchgrössen hinsichtlich ihrer Validierungsleistung und Trainingsdauer untersucht. Die übrigen Hyperparameter blieben unverändert. Alle Experimente verwendeten den Random Seed 42 und Early Stopping mit einer Patience von fünf Epochen.

| Batch Size | Beste Validation Accuracy | Beste Epoche | Trainingszeit |
|---|---:|---:|---:|
| 16 | 66.30 % | 6 | 65.2 s |
| 32 | 66.20 % | 8 | 43.8 s |
| 64 | 64.90 % | 12 | 32.9 s |
| 128 | 66.60 % | 17 | 30.9 s |
| **256** | **66.90 %** | **28** | 34.6 s |
| 512 | 65.80 % | 24 | 24.3 s |

Die Ergebnisse zeigen, dass grössere Batchgrössen tendenziell mehr Epochen benötigen, um ihre höchste Validierungsleistung zu erreichen. Dies lässt sich teilweise dadurch erklären, dass bei grösseren Batches weniger Gewichtsaktualisierungen pro Epoche durchgeführt werden.

Die höchste Validation Accuracy wurde mit **Batch Size 256 (66.90 %)** erreicht. Diese Konfiguration wird deshalb für die weiteren Experimente ausgewählt.

Da die Unterschiede zwischen mehreren Konfigurationen gering sind und die Experimente jeweils nur mit einem Random Seed durchgeführt wurden, ist die Auswahl als vorläufig zu betrachten.

Eine abschliessende Bewertung der Modellleistung erfolgt erst nach der Hyperparameteroptimierung auf dem bisher nicht verwendeten Testdatensatz.

### 2.25 Learning-Rate-Optimierung

Zur Optimierung des Modells werden die Learning Rates 0.01, 0.001 und 0.0001 bei einer Batch Size von 256 verglichen. Die übrigen Hyperparameter bleiben unverändert und die Ergebnisse werden in W&B protokolliert.

In [50]:
result_lr001 = run_experiment(
    batch_size=256,
    learning_rate=0.01,
    epochs=60,
    patience=5
)

BS 256 | Epoch 1/60 | Train Acc: 59.35% | Valid Acc: 61.80%
BS 256 | Epoch 2/60 | Train Acc: 71.84% | Valid Acc: 64.40%
BS 256 | Epoch 3/60 | Train Acc: 78.69% | Valid Acc: 65.40%
BS 256 | Epoch 4/60 | Train Acc: 83.92% | Valid Acc: 66.00%
BS 256 | Epoch 5/60 | Train Acc: 89.43% | Valid Acc: 65.10%
BS 256 | Epoch 6/60 | Train Acc: 94.17% | Valid Acc: 64.50%
BS 256 | Epoch 7/60 | Train Acc: 96.95% | Valid Acc: 64.70%
BS 256 | Epoch 8/60 | Train Acc: 98.52% | Valid Acc: 65.20%
BS 256 | Epoch 9/60 | Train Acc: 99.05% | Valid Acc: 65.30%
Early Stopping nach Epoche 9

Beste Validation Accuracy: 66.00%
Beste Epoche: 4
Trainingszeit: 10.1 Sekunden


epoch,▁▂▃▄▅▅▆▇█
train_accuracy,▁▃▄▅▆▇███
train_loss,█▇▆▅▃▂▂▁▁
valid_accuracy,▁▅▇█▇▅▆▇▇
valid_loss,▁▁▁▂▃▄▆▇█
best_epoch,4
best_valid_accuracy,0.66
epoch,9
train_accuracy,0.99054
train_loss,0.02807
training_time_seconds,10.05218


Failed to export metrics batch due to timeout, max retries or shutdown.


In [51]:
result_lr0001 = run_experiment(
    batch_size=256,
    learning_rate=0.0001,
    epochs=60,
    patience=5
)

BS 256 | Epoch 1/60 | Train Acc: 51.99% | Valid Acc: 50.80%
BS 256 | Epoch 2/60 | Train Acc: 53.63% | Valid Acc: 51.70%
BS 256 | Epoch 3/60 | Train Acc: 55.13% | Valid Acc: 52.80%
BS 256 | Epoch 4/60 | Train Acc: 55.73% | Valid Acc: 52.90%
BS 256 | Epoch 5/60 | Train Acc: 56.43% | Valid Acc: 51.60%
BS 256 | Epoch 6/60 | Train Acc: 56.93% | Valid Acc: 52.40%
BS 256 | Epoch 7/60 | Train Acc: 57.33% | Valid Acc: 53.10%
BS 256 | Epoch 8/60 | Train Acc: 57.57% | Valid Acc: 53.80%
BS 256 | Epoch 9/60 | Train Acc: 57.94% | Valid Acc: 53.80%
BS 256 | Epoch 10/60 | Train Acc: 58.24% | Valid Acc: 53.90%
BS 256 | Epoch 11/60 | Train Acc: 58.72% | Valid Acc: 54.70%
BS 256 | Epoch 12/60 | Train Acc: 59.07% | Valid Acc: 54.00%
BS 256 | Epoch 13/60 | Train Acc: 59.54% | Valid Acc: 54.30%
BS 256 | Epoch 14/60 | Train Acc: 59.94% | Valid Acc: 54.80%
BS 256 | Epoch 15/60 | Train Acc: 60.38% | Valid Acc: 55.50%
BS 256 | Epoch 16/60 | Train Acc: 60.72% | Valid Acc: 55.50%
BS 256 | Epoch 17/60 | Train Acc:

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
train_accuracy,▁▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇█████
train_loss,█████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
valid_accuracy,▁▂▂▂▂▃▃▃▄▃▄▄▄▄▅▅▅▆▆▅▆▆▆▆▆▇▆▇▇▇▇▇▇▇██████
valid_loss,███▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
best_epoch,47
best_valid_accuracy,0.61
epoch,52
train_accuracy,0.7002
train_loss,0.56935
training_time_seconds,56.77615


In [52]:
result_lr00316 = run_experiment(
    batch_size=256,
    learning_rate=10**(-2.5),
    epochs=60,
    patience=5
)

BS 256 | Epoch 1/60 | Train Acc: 56.75% | Valid Acc: 57.00%
BS 256 | Epoch 2/60 | Train Acc: 63.97% | Valid Acc: 61.50%
BS 256 | Epoch 3/60 | Train Acc: 70.44% | Valid Acc: 62.70%
BS 256 | Epoch 4/60 | Train Acc: 75.28% | Valid Acc: 64.80%
BS 256 | Epoch 5/60 | Train Acc: 79.30% | Valid Acc: 64.40%
BS 256 | Epoch 6/60 | Train Acc: 82.61% | Valid Acc: 64.10%
BS 256 | Epoch 7/60 | Train Acc: 85.80% | Valid Acc: 64.60%
BS 256 | Epoch 8/60 | Train Acc: 88.76% | Valid Acc: 65.00%
BS 256 | Epoch 9/60 | Train Acc: 91.30% | Valid Acc: 65.60%
BS 256 | Epoch 10/60 | Train Acc: 93.51% | Valid Acc: 64.70%
BS 256 | Epoch 11/60 | Train Acc: 95.17% | Valid Acc: 65.40%
BS 256 | Epoch 12/60 | Train Acc: 96.30% | Valid Acc: 65.40%
BS 256 | Epoch 13/60 | Train Acc: 97.36% | Valid Acc: 65.60%
BS 256 | Epoch 14/60 | Train Acc: 98.12% | Valid Acc: 64.90%
Early Stopping nach Epoche 14

Beste Validation Accuracy: 65.60%
Beste Epoche: 9
Trainingszeit: 15.6 Sekunden


epoch,▁▂▂▃▃▄▄▅▅▆▆▇▇█
train_accuracy,▁▂▃▄▅▅▆▆▇▇████
train_loss,█▇▇▆▅▅▄▃▃▂▂▁▁▁
valid_accuracy,▁▅▆▇▇▇▇██▇███▇
valid_loss,▁▁▁▁▁▂▂▃▄▅▅▆▇█
best_epoch,9
best_valid_accuracy,0.656
epoch,14
train_accuracy,0.98119
train_loss,0.07036
training_time_seconds,15.61219


In [53]:
result_lr000316 = run_experiment(
    batch_size=256,
    learning_rate=10**(-3.5),
    epochs=60,
    patience=5
)

BS 256 | Epoch 1/60 | Train Acc: 53.16% | Valid Acc: 51.60%
BS 256 | Epoch 2/60 | Train Acc: 56.07% | Valid Acc: 52.80%
BS 256 | Epoch 3/60 | Train Acc: 57.40% | Valid Acc: 53.30%
BS 256 | Epoch 4/60 | Train Acc: 58.56% | Valid Acc: 54.00%
BS 256 | Epoch 5/60 | Train Acc: 59.72% | Valid Acc: 55.90%
BS 256 | Epoch 6/60 | Train Acc: 60.86% | Valid Acc: 56.60%
BS 256 | Epoch 7/60 | Train Acc: 62.10% | Valid Acc: 56.90%
BS 256 | Epoch 8/60 | Train Acc: 63.00% | Valid Acc: 57.60%
BS 256 | Epoch 9/60 | Train Acc: 64.09% | Valid Acc: 57.70%
BS 256 | Epoch 10/60 | Train Acc: 64.69% | Valid Acc: 58.60%
BS 256 | Epoch 11/60 | Train Acc: 65.71% | Valid Acc: 59.40%
BS 256 | Epoch 12/60 | Train Acc: 66.59% | Valid Acc: 59.70%
BS 256 | Epoch 13/60 | Train Acc: 67.44% | Valid Acc: 60.10%
BS 256 | Epoch 14/60 | Train Acc: 68.36% | Valid Acc: 60.80%
BS 256 | Epoch 15/60 | Train Acc: 69.03% | Valid Acc: 61.00%
BS 256 | Epoch 16/60 | Train Acc: 70.01% | Valid Acc: 60.10%
BS 256 | Epoch 17/60 | Train Acc:

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
train_accuracy,▁▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█████
train_loss,████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁
valid_accuracy,▁▂▂▂▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇████▇██████
valid_loss,▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▆▆▇▇█
best_epoch,37
best_valid_accuracy,0.646
epoch,42
train_accuracy,0.83525
train_loss,0.3696
training_time_seconds,47.47151


### 2.27 Ergebnisse der Learning-Rate-Optimierung

Die Learning Rate wurde zunächst in exponentiellen Abständen und anschliessend mit logarithmischen Zwischenwerten untersucht. Alle Experimente verwendeten Batch Size 256 und Early Stopping.

| Learning Rate | Beste Validation Accuracy | Beste Epoche |
|---|---:|---:|
| 0.0001 | 61.00 % | 47 |
| 0.000316 | 64.60 % | 37 |
| **0.001** | **66.90 %** | 28 |
| 0.003162 | 65.60 % | 9 |
| 0.01 | 66.00 % | 4 |

Die höchste Validation Accuracy wurde mit einer Learning Rate von **0.001** erreicht. Diese wird für die weiteren Experimente beibehalten.

### 2.28 Dropout-Regularisierung

Zur Reduktion von Overfitting wird Dropout in das Word-Embedding-Modell integriert. Die Dropout-Raten 0.0, 0.2 und 0.5 werden miteinander verglichen.

In [54]:
import importlib
import src.models

importlib.reload(src.models)

WordEmbeddingModel = src.models.WordEmbeddingModel

test_model = WordEmbeddingModel(
    vocab_size=len(vocab),
    embedding_dim=50,
    dropout=0.2
)

print(test_model.classifier)

Sequential(
  (0): Linear(in_features=150, out_features=64, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=64, out_features=1, bias=True)
)


In [56]:
result_baseline_check = run_experiment(
    batch_size=256,
    learning_rate=0.001,
    epochs=60,
    patience=5,
    seed=42,
    dropout=0.0,
    weight_decay=0.01
)

TypeError: run_experiment() got an unexpected keyword argument 'dropout'